In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
    .appName("twibot")
    .config("spark.jars.packages",
            "graphframes:graphframes:0.8.3-spark3.5-s_2.12,"
            "org.mongodb.spark:mongo-spark-connector_2.12:10.4.0")
    .config("spark.mongodb.write.connection.uri", "mongodb://mongo:27017")
    .config("spark.mongodb.read.connection.uri", "mongodb://mongo:27017")
    .config("spark.driver.memory", "1g")
    .getOrCreate())
out = "/home/jovyan/work/parquet"
print(spark.version)

In [ ]:
from sklearn.metrics import roc_auc_score

lp = spark.read.parquet(f"{out}/communities").select(
        "id", "bot_label", F.col("label").alias("community"))
sc = spark.read.parquet(f"{out}/community_scores").select(
        F.col("label").alias("community"), "coord_score", "lift")
sp = spark.read.parquet(f"{out}/split")

d = lp.join(sc, "community").join(sp, "id").toPandas()
d["y"] = (d["bot_label"] == "bot").astype(int)

print(d.groupby("split").agg(users=("y", "size"), bots=("y", "sum")))
for s in sorted(d["split"].unique()):
    x = d[d["split"] == s]
    if x["y"].nunique() == 2:
        print(s, "AUC of coord_score:", round(roc_auc_score(x["y"], x["coord_score"]), 3))
        print(s, "AUC of community lift (uses labels, reference only):",
              round(roc_auc_score(x["y"], x["lift"]), 3))
        